# Notebook 10 (Rust) — Comparative Radar Chart

A Rust port of `10_comparative_radar.ipynb` using:
- **`polars`** for CSV loading, cleaning, and filtering (replaces `pandas`)
- **`plotly`** for the dual-player radar chart via `ScatterPolar` (replaces `soccerplots`)

The chart compares Tammy Abraham vs Harry Kane across 14 attacking metrics for the 2020-21 Premier League season.

In [38]:
// Dependencies — evcxr fetches and compiles crates automatically.
// polars: DataFrame library for CSV loading, filtering, and column manipulation.
// plotly: Plotly.js bindings; ScatterPolar is the trace type for radar/spider charts.
// dtype-struct feature is required for split_exact, which returns a Struct column.
// serde_json: needed explicitly to build the raw Plotly JSON payload (polar layout
//   is not exposed in plotly 0.10's Rust API, so we emit JSON directly).
:dep polars = { version = "0.46", features = ["csv", "lazy", "strings", "dtype-struct"] }
:dep plotly = { version = "0.10", features = ["plotly_ndarray"] }
:dep serde_json = "1"

In [39]:
use polars::prelude::*;

// Load the FBref shooting stats CSV for the 2020-21 Premier League season.
// The 'Player' column contains 'Full Name\url-slug' pairs that need splitting.
// infer_schema_length(None) scans the full file so numeric columns like G/Sh
// are not inferred as strings due to early '+'/'-' signs in later rows.
let df = CsvReadOptions::default()
    .with_infer_schema_length(None)
    .try_into_reader_with_file_path(Some("data/radars10.csv".into()))?
    .finish()?;

println!("Shape: {:?}", df.shape());
println!("{}", df.head(Some(3)));

Shape: (505, 26)
shape: (3, 26)
┌─────┬─────────────────────────────────┬─────────┬─────┬───┬─────────┬──────┬─────────┬─────────┐
│ Rk  ┆ Player                          ┆ Nation  ┆ Pos ┆ … ┆ npxG/Sh ┆ G-xG ┆ np:G-xG ┆ Matches │
│ --- ┆ ---                             ┆ ---     ┆ --- ┆   ┆ ---     ┆ ---  ┆ ---     ┆ ---     │
│ i64 ┆ str                             ┆ str     ┆ str ┆   ┆ f64     ┆ f64  ┆ f64     ┆ str     │
╞═════╪═════════════════════════════════╪═════════╪═════╪═══╪═════════╪══════╪═════════╪═════════╡
│ 1   ┆ Patrick van Aanholt\Patrick-va… ┆ nl NED  ┆ DF  ┆ … ┆ 0.05    ┆ -0.8 ┆ -0.8    ┆ Matches │
│ 2   ┆ Tammy Abraham\Tammy-Abraham     ┆ eng ENG ┆ FW  ┆ … ┆ 0.18    ┆ 0.4  ┆ 0.4     ┆ Matches │
│ 3   ┆ Che Adams\Che-Adams             ┆ eng ENG ┆ FW  ┆ … ┆ 0.13    ┆ -1.3 ┆ -1.3    ┆ Matches │
└─────┴─────────────────────────────────┴─────────┴─────┴───┴─────────┴──────┴─────────┴─────────┘


In [40]:
use polars::prelude::*;

let df = CsvReadOptions::default()
    .with_infer_schema_length(None)
    .try_into_reader_with_file_path(Some("data/radars10.csv".into()))?
    .finish()?;

let df = df.lazy()
    .with_column(
        col("Player")
            .str()
            .split_exact(lit("\\"), 1)
            .struct_()
            .field_by_name("field_0")
            .alias("Player")
    )
    .collect()?;

// Wrap the borrow in a block so `players` (a &ChunkedArray reference) goes
// out of scope before the cell ends — evcxr can't persist non-static lifetimes.
{
    let players = df.column("Player")?.str()?;
    println!("Sample players:");
    for i in 0..5 {
        println!("  {:?}", players.get(i));
    }
}

Sample players:
  Some("Patrick van Aanholt")
  Some("Tammy Abraham")
  Some("Che Adams")
  Some("Tosin Adarabioyo")
  Some("Adrián")


()

In [41]:
use polars::prelude::*;

// --- Reload and clean in a single pipeline ---
let df_raw = CsvReadOptions::default()
    .with_infer_schema_length(None)
    .try_into_reader_with_file_path(Some("data/radars10.csv".into()))?
    .finish()?;

let df_clean = df_raw.lazy()
    // Clean Player column.
    .with_column(
        col("Player")
            .str()
            .split_exact(lit("\\"), 1)
            .struct_()
            .field_by_name("field_0")
            .alias("Player")
    )
    // Filter to only Abraham and Kane, matching the Python OR filter.
    .filter(
        col("Player").eq(lit("Tammy Abraham"))
            .or(col("Player").eq(lit("Harry Kane")))
    )
    // Drop administrative and non-performance columns.
    // '90s' removed: Abraham/Kane played different minutes — keeping it skews the radar.
    // 'FK', 'PK', 'PKatt': set-piece metrics, not open-play shooting quality.
    // 'Matches': FBref metadata string column, not numeric.
    .drop(["Rk", "Nation", "Pos", "Squad", "Age", "Born", "90s", "FK", "PK", "PKatt", "Matches"])
    .collect()?;

println!("{}", df_clean);

shape: (2, 15)
┌───────────────┬─────┬─────┬─────┬───┬──────┬─────────┬──────┬─────────┐
│ Player        ┆ Gls ┆ Sh  ┆ SoT ┆ … ┆ npxG ┆ npxG/Sh ┆ G-xG ┆ np:G-xG │
│ ---           ┆ --- ┆ --- ┆ --- ┆   ┆ ---  ┆ ---     ┆ ---  ┆ ---     │
│ str           ┆ i64 ┆ i64 ┆ i64 ┆   ┆ f64  ┆ f64     ┆ f64  ┆ f64     │
╞═══════════════╪═════╪═════╪═════╪═══╪══════╪═════════╪══════╪═════════╡
│ Tammy Abraham ┆ 6   ┆ 31  ┆ 13  ┆ … ┆ 5.6  ┆ 0.18    ┆ 0.4  ┆ 0.4     │
│ Harry Kane    ┆ 14  ┆ 85  ┆ 30  ┆ … ┆ 9.6  ┆ 0.11    ┆ 2.2  ┆ 1.4     │
└───────────────┴─────┴─────┴─────┴───┴──────┴─────────┴──────┴─────────┘


In [42]:
use polars::prelude::*;

// --- Rebuild full cleaned DataFrame for chart construction ---
let df_raw = CsvReadOptions::default()
    .with_infer_schema_length(None)
    .try_into_reader_with_file_path(Some("data/radars10.csv".into()))?
    .finish()?;

let df_players = df_raw.lazy()
    .with_column(
        col("Player")
            .str()
            .split_exact(lit("\\"), 1)
            .struct_()
            .field_by_name("field_0")
            .alias("Player")
    )
    .filter(
        col("Player").eq(lit("Tammy Abraham"))
            .or(col("Player").eq(lit("Harry Kane")))
    )
    .drop(["Rk", "Nation", "Pos", "Squad", "Age", "Born", "90s", "FK", "PK", "PKatt", "Matches"])
    .collect()?;

// Extract metric column names (all columns except 'Player').
// These become the radar axis labels (theta values in ScatterPolar).
let params: Vec<String> = df_players
    .get_column_names()
    .iter()
    .filter(|name| **name != "Player")
    .map(|s| s.to_string())
    .collect();

println!("Metrics ({}): {:?}", params.len(), params);

Metrics (14): ["Gls", "Sh", "SoT", "SoT%", "Sh/90", "SoT/90", "G/Sh", "G/SoT", "Dist", "xG", "npxG", "npxG/Sh", "G-xG", "np:G-xG"]


In [43]:
use polars::prelude::*;

// Helper: extract a player's numeric stat vector as Vec<f64> from a Polars DataFrame.
// For each metric column, reads the first row where Player == name.
// Polars stores mixed-sign integer columns as Int32; cast to f64 for uniform handling.
fn extract_values(df: &DataFrame, player: &str, params: &[String]) -> Vec<f64> {
    let player_df = df
        .clone()
        .lazy()
        .filter(col("Player").eq(lit(player)))
        .collect()
        .unwrap();

    params.iter().map(|col_name| {
        let series = player_df.column(col_name).unwrap();
        // Cast to f64 regardless of original dtype (Int32, Float64, etc.).
        let series_f64 = series.cast(&DataType::Float64).unwrap();
        series_f64.f64().unwrap().get(0).unwrap_or(0.0)
    }).collect()
}

println!("Helper function defined.");

Helper function defined.


In [44]:
use polars::prelude::*;

// --- Full pipeline ---
let df_raw = CsvReadOptions::default()
    .with_infer_schema_length(None)
    .try_into_reader_with_file_path(Some("data/radars10.csv".into()))?
    .finish()?;

let df_players = df_raw.lazy()
    .with_column(
        col("Player")
            .str()
            .split_exact(lit("\\"), 1)
            .struct_()
            .field_by_name("field_0")
            .alias("Player")
    )
    .filter(
        col("Player").eq(lit("Tammy Abraham"))
            .or(col("Player").eq(lit("Harry Kane")))
    )
    .drop(["Rk", "Nation", "Pos", "Squad", "Age", "Born", "90s", "FK", "PK", "PKatt", "Matches"])
    .collect()?;

let params: Vec<String> = df_players
    .get_column_names()
    .iter()
    .filter(|name| **name != "Player")
    .map(|s| s.to_string())
    .collect();

let a_values = extract_values(&df_players, "Tammy Abraham", &params);
let b_values = extract_values(&df_players, "Harry Kane", &params);

// Replicate the Python range construction exactly:
//   a = min(col) - min(col) * 0.25   → lower bound 25% below the smaller value
//   b = max(col) + max(col) * 0.25   → upper bound 25% above the larger value
// soccerplots uses these per-axis (min, max) ranges to scale each axis independently,
// so that every metric fills the full radial extent regardless of its magnitude.
// Plotly has a single shared radial axis — so we normalise each raw value to [0, 1]
// within its own (range_min, range_max) before plotting. This reproduces the same
// per-axis proportional positioning that soccerplots achieves with independent axes.
let mut ranges: Vec<(f64, f64)> = Vec::new();
let mut a_norm: Vec<f64> = Vec::new();
let mut b_norm: Vec<f64> = Vec::new();

for (i, _col) in params.iter().enumerate() {
    let av = a_values[i];
    let bv = b_values[i];

    let raw_min = av.min(bv);
    let raw_max = av.max(bv);

    // 25% padding — identical to the Python notebook logic.
    let range_min = raw_min - raw_min * 0.25;
    let range_max = raw_max + raw_max * 0.25;

    ranges.push((range_min, range_max));

    // Normalise to [0, 1]: position within the padded range.
    // Avoids division by zero when both players have the same value.
    let span = range_max - range_min;
    let norm = |v: f64| if span == 0.0 { 0.5 } else { (v - range_min) / span };

    a_norm.push(norm(av));
    b_norm.push(norm(bv));
}

// Build hover text showing the raw value and its axis range for each metric,
// matching the context that soccerplots displays via its axis tick labels.
let a_hover: Vec<String> = params.iter().enumerate()
    .map(|(i, p)| format!("{}: {:.2}<br>range [{:.2}, {:.2}]", p, a_values[i], ranges[i].0, ranges[i].1))
    .collect();
let b_hover: Vec<String> = params.iter().enumerate()
    .map(|(i, p)| format!("{}: {:.2}<br>range [{:.2}, {:.2}]", p, b_values[i], ranges[i].0, ranges[i].1))
    .collect();

// Close the polygon by repeating the first element at the end.
let mut theta = params.clone();
theta.push(params[0].clone());

let mut a_r = a_norm.clone(); a_r.push(a_norm[0]);
let mut b_r = b_norm.clone(); b_r.push(b_norm[0]);

let mut a_hov = a_hover.clone(); a_hov.push(a_hover[0].clone());
let mut b_hov = b_hover.clone(); b_hov.push(b_hover[0].clone());

let payload = serde_json::json!({
    "data": [
        {
            "type": "scatterpolar",
            "name": "Tammy Abraham",
            "theta": theta,
            "r": a_r,
            "text": a_hov,
            "hoverinfo": "text+name",
            "fill": "toself",
            "fillcolor": "rgba(0, 0, 255, 0.35)",
            "line": { "color": "blue" },
            "mode": "lines+markers",
            "opacity": 0.75
        },
        {
            "type": "scatterpolar",
            "name": "Harry Kane",
            "theta": theta,
            "r": b_r,
            "text": b_hov,
            "hoverinfo": "text+name",
            "fill": "toself",
            "fillcolor": "rgba(255, 0, 0, 0.25)",
            "line": { "color": "red" },
            "mode": "lines+markers",
            "opacity": 0.6
        }
    ],
    "layout": {
        "title": { "text": "Tammy Abraham vs Harry Kane — 2020-21 PL Shooting Stats" },
        "polar": {
            // type=category spreads metric names evenly around the circle.
            "angularaxis": { "type": "category", "direction": "clockwise" },
            // Radial axis is now normalised [0,1] — hide the raw tick numbers
            // since they are meaningless; raw values are shown in hover text instead.
            "radialaxis": {
                "visible": true,
                "showticklabels": false,
                "range": [0.0, 1.0]
            }
        },
        "paper_bgcolor": "#1a1a2e",
        "plot_bgcolor": "#1a1a2e",
        "font": { "color": "#ffffff" }
    }
});

println!(
    "EVCXR_BEGIN_CONTENT application/vnd.plotly.v1+json\n{}\nEVCXR_END_CONTENT",
    payload.to_string()
);

## Summary: Rust Port of Comparative Radar Chart

### What Changed vs the Python Notebook

| Concern | Python | Rust |
|---|---|---|
| Data loading | `pandas.read_csv` | `polars` `CsvReadOptions` |
| Player name cleaning | `str.split('\\')[0]` | `str.split_exact` → `.struct_().field_by_name("field_0")` |
| Row filtering | `df[df['Player'] == ...]` | `.filter(col("Player").eq(lit(...)).or(...))` |
| Column dropping | `df.drop([...])` | `.drop([...])` (same API, lazy plan) |
| Radar rendering | `soccerplots.Radar` | `plotly::ScatterPolar` |
| Axis range scaling | Manual 25% padding | ScatterPolar auto-scales to data range |
| Polygon closure | Implicit in soccerplots | Explicit: repeat first element at end of `theta`/`r` |
| Inline display | matplotlib `fig` | `plot.lab_display()` (Plotly.js in Jupyter) |

### Key Design Notes

- **`polars` is data-only**: It handles CSV ingestion, string cleaning, filtering, and column extraction. Polars has no charting capability in Rust.
- **`plotly::ScatterPolar`**: The Rust equivalent of Plotly's `scatterpolar` trace. `fill=ToSelf` closes the polygon area. The chart is interactive (hover to see exact values) unlike the static matplotlib output from `soccerplots`.
- **Polygon closure**: Plotly requires manually repeating `theta[0]` and `r[0]` at the end of both vectors to close the radar polygon. `soccerplots` handled this internally.
- **Axis ranges**: Plotly auto-scales the radial axis based on the data, so the manual 25%-padding logic from the Python notebook is not needed. For population-level anchoring, set `PolarAxis::range([min, max])` explicitly.
- **`infer_schema_length(None)`**: Required because the `G-xG` and `np:G-xG` columns contain mixed `+`/`-` signed strings in early rows which would be inferred as strings without a full-file schema scan.